# VQA su Flickr30k 
---

## **TRAINING & FINE-TUNING**

## STEP 1 — Installing libraries

In [1]:

!pip install -q --upgrade peft transformers 
!pip install -q "Pillow<12.0"
!pip install -q bitsandbytes
!pip install -q accelerate
!pip install -q -U gradio
!pip install -q datasets==2.19.1
!pip install -q openai
!pip install -q evaluate
!pip install -q bert_score  

print('Librerie installate correttamente!')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 680.7/680.7 kB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 102.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 113.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]

## STEP 2 — Import e setup folders

In [2]:
import os
import json
import random
import numpy as np
import torch
from PIL import Image
from tqdm import tqdm

# HuggingFace
from datasets import load_dataset
from transformers import (
    CLIPProcessor,
    CLIPModel,
    LlavaForConditionalGeneration,
    AutoProcessor,
    BitsAndBytesConfig
)
from peft import LoraConfig, get_peft_model, PeftModel

# PyTorch
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

print(f'PyTorch version: {torch.__version__}')
print(f'GPU available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

PyTorch version: 2.10.0+cu128
GPU available: True
GPU: Tesla T4
VRAM: 15.6 GB


In [3]:
# Create the structure of folders
BASE_PATH = '/kaggle/working'

DIRS = {
    'vqa_pairs':    os.path.join(BASE_PATH, 'vqa_pairs'),
    'features':     os.path.join(BASE_PATH, 'features'),
    'checkpoints':  os.path.join(BASE_PATH, 'checkpoints'),
    'final_model_v2':  os.path.join(BASE_PATH, 'final_model_v2'),
}

for name, path in DIRS.items():
    os.makedirs(path, exist_ok=True)
    print(f'Folder created: {path}')

# Constants of the project
MODEL_NAME      = 'llava-hf/llava-1.5-7b-hf'
CLIP_MODEL_NAME = 'openai/clip-vit-large-patch14'
DEVICE          = 'cuda' if torch.cuda.is_available() else 'cpu'
MAX_SAMPLES     = 4000   # if None, it will use all the dataset
BATCH_SIZE      = 1
ACCUMULATION_STEPS = 4
N_EPOCHS        = 1
LEARNING_RATE   = 2e-5

print(f'\nDevice: {DEVICE}')
print(f'Modello: {MODEL_NAME}')

Folder created: /kaggle/working/vqa_pairs
Folder created: /kaggle/working/features
Folder created: /kaggle/working/checkpoints
Folder created: /kaggle/working/final_model_v2

Device: cuda
Modello: llava-hf/llava-1.5-7b-hf


## STEP 3 — Loading Flickr30k

In [4]:
def load_flickr30k(max_samples=None):
    """
    Carica il dataset Flickr30k da HuggingFace.
    Ritorna un dizionario con split train e test.
    """
    print('Caricamento Flickr30k...')
    
    # Carica il dataset completo
    dataset = load_dataset('nlphuji/flickr30k', trust_remote_code=True)
    
    # Flickr30k ha solo uno split 'test', quindi lo dividiamo manualmente
    full_data = dataset['test']
    
    # Limita i campioni se richiesto (utile per debug)
    if max_samples is not None:
        full_data = full_data.select(range(min(max_samples, len(full_data))))
    
    # Split 90% train, 10% test
    split = full_data.train_test_split(test_size=0.1, seed=42)
    
    print(f'Train: {len(split["train"])} esempi')
    print(f'Test:  {len(split["test"])} esempi')
    
    return split


def inspect_sample(dataset, idx=0):
    """
    Stampa un campione del dataset per capirne la struttura.
    """
    sample = dataset[idx]
    print(f'Chiavi disponibili: {list(sample.keys())}')
    print(f'Tipo immagine: {type(sample["image"])}')
    print(f'Dimensione immagine: {sample["image"].size}')
    print(f'Caption (ce ne sono {len(sample["caption"])}):' )
    for i, cap in enumerate(sample['caption']):
        print(f'  [{i}] {cap}')


# Carica il dataset
dataset = load_flickr30k(max_samples=MAX_SAMPLES)

print('\n--- Esempio campione ---')
inspect_sample(dataset['train'], idx=0)

Caricamento Flickr30k...


Generating test split:   0%|          | 0/31014 [00:00<?, ? examples/s]

Train: 3600 esempi
Test:  400 esempi

--- Esempio campione ---
Chiavi disponibili: ['image', 'caption', 'sentids', 'split', 'img_id', 'filename']
Tipo immagine: <class 'PIL.JpegImagePlugin.JpegImageFile'>
Dimensione immagine: (500, 385)
Caption (ce ne sono 5):
  [0] A woman wearing a white shirt is standing behind a counter.
  [1] A Woman working a food stand with lots of desserts.
  [2] The lady is white is selling things on the street.
  [3] A lady stands at her ice cream stand.
  [4] A woman standing in her food stand.


## STEP 4 — Generazione coppie VQA dalle caption (DeepSeek V4 + fallback rule-based)

In [5]:
# from openai import OpenAI
# import time

# # ─────────────────────────────────────────────
# # CONFIGURAZIONE DEEPSEEK V4

# # Importa la funzione speciale di Kaggle per leggere i Secret
# from kaggle_secrets import UserSecretsClient

# # Estrai la tua chiave (Assicurati che la Label nei secret si chiami esattamente DEEPSEEK_API_KEY)
# user_secrets = UserSecretsClient()
# mia_chiave_deepseek = user_secrets.get_secret("DEEPSEECK_API")

# # Inizializza il client passandogli la chiave estratta
# deepseek_client = OpenAI(
#     api_key = mia_chiave_deepseek,
#     base_url = 'https://api.deepseek.com'
# )

# # Prompt di sistema fisso — definisce il comportamento del modello
# # per tutta la generazione delle coppie Q&A
# SYSTEM_PROMPT = """
# You are an expert at generating Visual Question Answering (VQA) datasets.
# Given an image caption, you generate diverse and natural question-answer pairs.
# You always return ONLY a valid JSON array, with no additional text, no markdown, no backticks.
# Each element has exactly two fields: 'question' and 'answer'.
# """


# def caption_to_qa_pairs_deepseek(caption: str, max_retries: int = 3) -> list:
#     """
#     Genera coppie Q&A da una caption usando DeepSeek V4 Flash.
    
#     Parametri:
#         caption:     la caption dell'immagine
#         max_retries: numero di tentativi in caso di errore o JSON invalido
    
#     Ritorna una lista di dizionari {'question': ..., 'answer': ...}
#     In caso di errore ritorna una coppia generica come fallback.
#     """
    
#     user_prompt = f"""Caption: "{caption}"

# Generate exactly 5 diverse question-answer pairs based ONLY on this caption.
# Cover different aspects: actions, locations, colors, objects, people, quantities.
# Do not hallucinate or invent details that are not present in the text.
# Answers must be short (max 10 words) and directly answerable from the caption.

# Return ONLY a valid JSON array like this:
# [
#   {{"question": "What is the man doing?", "answer": "Reading a newspaper"}},
#   {{"question": "Where is the scene taking place?", "answer": "In a park"}}
# ]"""

#     for attempt in range(max_retries):
#         try:
#             response = deepseek_client.chat.completions.create(
#                 model='deepseek-v4-flash',
#                 messages=[
#                     {'role': 'system', 'content': SYSTEM_PROMPT},
#                     {'role': 'user',   'content': user_prompt}
#                 ],
#                 temperature=0.7,      # Un po' di variabilità nelle domande generate
#                 max_tokens=512,       # Le coppie Q&A sono corte, 512 è più che sufficiente
#             )
            
#             # Estrai il testo della risposta
#             raw = response.choices[0].message.content.strip()
            
#             # Rimuovi eventuali backtick markdown che il modello potrebbe aggiungere
#             # nonostante le istruzioni (es. ```json ... ```)
#             raw = raw.replace('```json', '').replace('```', '').strip()
            
#             # Parsa il JSON
#             pairs = json.loads(raw)
            
#             # Valida che sia una lista di dizionari con i campi corretti
#             if not isinstance(pairs, list):
#                 raise ValueError('Il modello non ha restituito una lista JSON')
            
#             valid_pairs = [
#                 p for p in pairs
#                 if isinstance(p, dict)
#                 and 'question' in p
#                 and 'answer' in p
#                 and len(p['question']) > 0
#                 and len(p['answer']) > 0
#             ]
            
#             if len(valid_pairs) == 0:
#                 raise ValueError('Nessuna coppia valida nel JSON restituito')
            
#             return valid_pairs
        
#         except json.JSONDecodeError:
#             # Il modello ha restituito testo non parsabile come JSON
#             # Riprova dopo una breve pausa
#             if attempt < max_retries - 1:
#                 time.sleep(1)
#             continue
        
#         except Exception as e:
#             # Errore di rete, rate limit, ecc.
#             # Se è l'ultimo tentativo usa il fallback
#             if attempt == max_retries - 1:
#                 break
#             # Altrimenti aspetta un po' di più (exponential backoff)
#             time.sleep(2 ** attempt)
    
#     # Fallback — se tutti i tentativi falliscono restituisce almeno una coppia generica
#     print(f'  [WARN] Fallback su caption: {caption[:60]}...')
#     return [{'question': 'What is happening in this image?', 'answer': caption}]


# def build_vqa_dataset_deepseek(flickr_split, split_name: str) -> list:
#     """
#     Costruisce il dataset VQA completo usando DeepSeek V4 Flash.
    
#     Strategia di salvataggio incrementale:
#     Salva ogni 100 immagini processate su disco.
#     Così se la sessione si interrompe non perdi tutto il lavoro.
    
#     Alla prossima esecuzione riprende dall'ultimo checkpoint.
#     """
    
#     # Path del file di output e del checkpoint di progresso
#     output_path     = os.path.join(DIRS['vqa_pairs'], f'{split_name}_vqa.json')
#     checkpoint_path = os.path.join(DIRS['vqa_pairs'], f'{split_name}_checkpoint.json')
    
#     # Controlla se esiste già un checkpoint parziale
#     # (sessione precedente interrotta a metà)
#     if os.path.exists(checkpoint_path):
#         with open(checkpoint_path, 'r') as f:
#             checkpoint = json.load(f)
#         start_idx = checkpoint['last_idx'] + 1
#         vqa_data  = checkpoint['data']
#         print(f'Ripreso da checkpoint: già processate {start_idx} immagini su {len(flickr_split)}')
#     else:
#         start_idx = 0
#         vqa_data  = []
#         print(f'Inizio generazione VQA per {split_name} ({len(flickr_split)} immagini)...')
    
#     SAVE_EVERY = 100  # Salva su disco ogni N immagini processate
    
#     for idx in tqdm(range(start_idx, len(flickr_split)), desc=f'DeepSeek VQA {split_name}'):
#         sample   = flickr_split[idx]
#         # Prendiamo solo la PRIMA caption per velocizzare di 5 volte il processo
#         caption = sample['caption'][0] 
        
#         pairs = caption_to_qa_pairs_deepseek(caption)
#         for pair in pairs:
#             vqa_data.append({
#                 'image_idx': idx,
#                 'question':  pair['question'],
#                 'answer':    pair['answer'],
#                 'caption':   caption
#             })
        
#         # Salvataggio incrementale ogni SAVE_EVERY immagini
#         if (idx + 1) % SAVE_EVERY == 0:
#             # Aggiorna il checkpoint
#             with open(checkpoint_path, 'w') as f:
#                 json.dump({'last_idx': idx, 'data': vqa_data}, f)
#             print(f'  Checkpoint salvato a idx={idx} ({len(vqa_data)} coppie totali)')
        
#         # Piccola pausa per rispettare i rate limit dell'API
#         # DeepSeek Flash è veloce ma ha limiti di chiamate al minuto
#         time.sleep(0.1)
    
#     # Salvataggio finale
#     with open(output_path, 'w') as f:
#         json.dump(vqa_data, f, indent=2)
#     print(f'Dataset salvato: {output_path} ({len(vqa_data)} coppie totali)')
    
#     # Rimuovi il checkpoint ora che abbiamo il file finale
#     if os.path.exists(checkpoint_path):
#         os.remove(checkpoint_path)
    
#     return vqa_data


# def load_vqa_pairs(split_name: str) -> list:
#     """
#     Carica le coppie VQA salvate su disco.
#     """
#     path = os.path.join(DIRS['vqa_pairs'], f'{split_name}_vqa.json')
#     with open(path, 'r') as f:
#         return json.load(f)


# # ─────────────────────────────────────────────
# # TEST RAPIDO — verifica che DeepSeek funzioni
# # prima di lanciare la generazione completa
# # ─────────────────────────────────────────────
# print('Test DeepSeek V4 Flash su una caption di esempio...')
# test_caption = 'A young woman in a red dress is dancing on a stage in front of a large crowd.'
# test_pairs = caption_to_qa_pairs_deepseek(test_caption)
# print(f'Caption: {test_caption}')
# print(f'Coppie generate ({len(test_pairs)}):')
# for p in test_pairs:
#     print(f'  Q: {p["question"]}')
#     print(f'  A: {p["answer"]}')
#     print()


# # ─────────────────────────────────────────────
# # GENERAZIONE COMPLETA
# # Se il test sopra funziona, esegui questa cella
# # ATTENZIONE: richiede ~2-4 ore per tutto il dataset
# # Lasciala girare di notte
# # ─────────────────────────────────────────────
# train_vqa_path = os.path.join(DIRS['vqa_pairs'], 'train_vqa.json')
# test_vqa_path  = os.path.join(DIRS['vqa_pairs'], 'test_vqa.json')

# if not os.path.exists(train_vqa_path):
#     train_vqa = build_vqa_dataset_deepseek(dataset['train'], 'train')
# else:
#     print('Coppie VQA train già presenti, le carico...')
#     train_vqa = load_vqa_pairs('train')

# if not os.path.exists(test_vqa_path):
#     test_vqa = build_vqa_dataset_deepseek(dataset['test'], 'test')
# else:
#     print('Coppie VQA test già presenti, le carico...')
#     test_vqa = load_vqa_pairs('test')

# print(f'\nTotale coppie train: {len(train_vqa)}')
# print(f'Totale coppie test:  {len(test_vqa)}')
# print(f'\nEsempio coppia generata da DeepSeek:')
# print(json.dumps(train_vqa[0], indent=2))

In [6]:


TRAIN_JSON_PATH = '/kaggle/input/datasets/lorenzobove24/llava-flickr-vqa-ready/train_vqa.json'
TEST_JSON_PATH  = '/kaggle/input/datasets/lorenzobove24/llava-flickr-vqa-ready/test_checkpoint.json'

print("Caricamento dei dati VQA pre-generati in corso...")

# 1. Carichiamo il Train Set
with open(TRAIN_JSON_PATH, 'r') as f:
    train_vqa_completo = json.load(f)

# Prendiamo solo 4000 campioni casuali!
NUMERO_CAMPIONI_TRAIN = 4000
train_vqa = random.sample(train_vqa_completo, min(NUMERO_CAMPIONI_TRAIN, len(train_vqa_completo)))

# 2. Carichiamo il Test Set
with open(TEST_JSON_PATH, 'r') as f:
    test_data_raw = json.load(f)
    if isinstance(test_data_raw, dict) and 'data' in test_data_raw:
        test_vqa = test_data_raw['data']
    else:
        test_vqa = test_data_raw

print(f"\n[OK] Dataset caricato e campionato! ")
print(f"Totale coppie train originali: {len(train_vqa_completo)}")
print(f"Totale coppie train pronte per la GPU: {len(train_vqa)} (Tempo stimato: ~2.5 ore)")
print(f"Totale coppie test pronte:  {len(test_vqa)}")

Caricamento dei dati VQA pre-generati in corso...

[OK] Dataset caricato e campionato! 
Totale coppie train originali: 14760
Totale coppie train pronte per la GPU: 4000 (Tempo stimato: ~2.5 ore)
Totale coppie test pronte:  800


## STEP 6 — Dataset PyTorch e DataLoader

In [7]:
class VQADataset(Dataset):
    """
    Dataset PyTorch per il fine-tuning di LLaVA su VQA.
    
    Ogni elemento restituisce:
    - pixel_values: tensore dell'immagine preprocessata
    - input_ids: token del prompt testuale
    - attention_mask: maschera di attenzione
    - labels: token della risposta attesa (per calcolare la loss)
    """
    
    def __init__(self, vqa_pairs: list, flickr_split, processor, max_length=512):
        """
        vqa_pairs:    lista di dizionari con 'image_idx', 'question', 'answer'
        flickr_split: il dataset Flickr30k (per recuperare le immagini)
        processor:    AutoProcessor di LLaVA
        max_length:   lunghezza massima della sequenza in token
        """
        self.vqa_pairs    = vqa_pairs
        self.flickr_split = flickr_split
        self.processor    = processor
        self.max_length   = max_length
    
    def __len__(self):
        return len(self.vqa_pairs)
    
    def __getitem__(self, idx):
        pair = self.vqa_pairs[idx]
        
        # Recupera l'immagine dal dataset Flickr30k
        image = self.flickr_split[pair['image_idx']]['image']
        
        # Costruisci il prompt nel formato che LLaVA si aspetta
        # Il token <image> indica dove va inserita l'immagine
        prompt = f"USER: <image>\n{pair['question']}\nASSISTANT: {pair['answer']}"
        
        # Preprocessa con il processor di LLaVA
        # Gestisce sia il testo che l'immagine insieme
        encoding = self.processor(
        images=image,
        text=prompt,
        return_tensors="pt",
        padding="max_length",
        truncation=True,
        max_length=1024  
)
        
        # Rimuovi la dimensione del batch aggiunta da return_tensors='pt'
        input_ids      = encoding['input_ids'].squeeze(0)
        attention_mask = encoding['attention_mask'].squeeze(0)
        pixel_values   = encoding['pixel_values'].squeeze(0)
        
        # I labels sono uguali agli input_ids per il language modeling
        # Ma mettiamo -100 sui token del prompt (solo la risposta contribuisce alla loss)
        labels = input_ids.clone()
        
        # Trova dove inizia la risposta dell'ASSISTANT nel prompt
        # e metti -100 su tutto il prompt (il modello non deve 'imparare' il prompt)
        assistant_token = self.processor.tokenizer.encode('ASSISTANT:', add_special_tokens=False)
        assistant_pos = self._find_subsequence(input_ids.tolist(), assistant_token)
        if assistant_pos != -1:
            labels[:assistant_pos + len(assistant_token)] = -100

        # Metti -100 su tutti i token di padding (gli spazi vuoti finali fino a 1024)
        pad_token_id = self.processor.tokenizer.pad_token_id
        # Se il tokenizzatore non ha un pad_token esplicito, usa eos_token
        if pad_token_id is None:
             pad_token_id = self.processor.tokenizer.eos_token_id
             
        labels[labels == pad_token_id] = -100
        
        
        return {
            'input_ids':      input_ids,
            'attention_mask': attention_mask,
            'pixel_values':   pixel_values,
            'labels':         labels
        }
    
    def _find_subsequence(self, sequence: list, subsequence: list) -> int:
        """Trova la posizione di una sottosequenza in una sequenza."""
        n = len(subsequence)
        for i in range(len(sequence) - n + 1):
            if sequence[i:i+n] == subsequence:
                return i
        return -1


def create_dataloaders(train_vqa, test_vqa, flickr_dataset, processor, batch_size=4):
    """
    Crea i DataLoader per train e test.
    """
    train_dataset = VQADataset(train_vqa, flickr_dataset['train'], processor)
    test_dataset  = VQADataset(test_vqa,  flickr_dataset['test'],  processor)
    
    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=2,
        pin_memory=True  # Velocizza il trasferimento su GPU
    )
    
    test_loader = DataLoader(
        test_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=2,
        pin_memory=True
    )
    
    print(f'Train loader: {len(train_loader)} batch da {batch_size}')
    print(f'Test loader:  {len(test_loader)} batch da {batch_size}')
    
    return train_loader, test_loader


print('Classi Dataset e DataLoader definite correttamente!')

Classi Dataset e DataLoader definite correttamente!


## STEP 7 — Carica LLaVA e applica LoRA

In [8]:
def load_processor(model_name=MODEL_NAME):
    """
    Carica il processor di LLaVA che gestisce
    sia il testo che le immagini.
    """
    print(f'Caricamento processor: {model_name}')
    processor = AutoProcessor.from_pretrained(model_name)
    
    # Imposta il padding token se non presente
    if processor.tokenizer.pad_token is None:
        processor.tokenizer.pad_token = processor.tokenizer.eos_token
    
    return processor


def load_base_model(model_name=MODEL_NAME):
    """
    Carica LLaVA in 4-bit quantization per ridurre
    l'uso di VRAM da ~28GB a ~5GB.
    """
    print(f'Caricamento modello: {model_name}')
    print('Questo richiede qualche minuto...')
    
    # Configurazione 4-bit quantization
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,               # Carica in 4-bit
        bnb_4bit_quant_type='nf4',       # Tipo di quantizzazione NF4
        bnb_4bit_compute_dtype=torch.float16,  # Calcoli in float16
        bnb_4bit_use_double_quant=True   # Doppia quantizzazione per risparmiare ancora memoria
    )
    
    model = LlavaForConditionalGeneration.from_pretrained(
        model_name,
        quantization_config=bnb_config,
        device_map='auto',   # Distribuisce automaticamente sui dispositivi disponibili
        torch_dtype=torch.float16
    )
    
    print(f'Modello caricato! Parametri totali: {sum(p.numel() for p in model.parameters()):,}')
    return model


def apply_lora(model):
    """
    Applica LoRA al modello per il fine-tuning efficiente.
    Solo i parametri LoRA vengono allenati (~1% del totale).
    """
    lora_config = LoraConfig(
        r=4,                    # Rango della decomposizione (più alto = più capacità ma più lento)
        lora_alpha=8,           # Scaling factor (di solito 2*r)
        lora_dropout=0.1,        # Dropout per regolarizzazione
        bias='none',             # Non aggiunge bias LoRA
        task_type='CAUSAL_LM',   # Tipo di task: language modeling causale
        # Questi sono i layer dell'LLM su cui applicare LoRA
        # Sono i layer di attenzione del transformer
        target_modules=[
            'q_proj',   # Query projection
            'v_proj',   # Value projection
            'k_proj',   # Key projection
            'o_proj',   # Output projection
        ]
    )
    
    # Applica LoRA al modello
    peft_model = get_peft_model(model, lora_config)

    return peft_model


def print_trainable_params(model):
    """
    Stampa quanti parametri sono effettivamente allenabili.
    Con LoRA dovrebbe essere circa l'1% del totale.
    """
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total     = sum(p.numel() for p in model.parameters())
    percent   = 100 * trainable / total
    
    print(f'Parametri trainabili: {trainable:,}')
    print(f'Parametri totali:     {total:,}')
    print(f'Percentuale:          {percent:.2f}%')


# --- Carica il modello ---
processor = load_processor()
base_model = load_base_model()
model = apply_lora(base_model)

print('\n--- Parametri trainabili con LoRA ---')
print_trainable_params(model)

Caricamento processor: llava-hf/llava-1.5-7b-hf


processor_config.json:   0%|          | 0.00/173 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/701 [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/674 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/505 [00:00<?, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/686 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/141 [00:00<?, ?B/s]

Modello caricato! Parametri totali: 3,663,943,680

--- Parametri trainabili con LoRA ---
Parametri trainabili: 4,784,128
Parametri totali:     3,668,727,808
Percentuale:          0.13%


In [9]:
# Crea i DataLoader
train_loader, test_loader = create_dataloaders(
    train_vqa, test_vqa, dataset, processor, batch_size=BATCH_SIZE
)

Train loader: 4000 batch da 1
Test loader:  800 batch da 1


## STEP 8 — Training Loop

In [10]:
def train_one_epoch(model, dataloader, optimizer, epoch_num):
    model.train()
    total_loss = 0.0
    n_batches  = 0
    
    optimizer.zero_grad()  # UNA SOLA VOLTA qui fuori dal loop
    
    # 1. Creiamo la barra di caricamento
    progress = tqdm(dataloader, desc=f'Epoch {epoch_num} [Train]')
    
    # 2. Iteriamo usando la variabile appena creata
    for i, batch in enumerate(progress):
        input_ids      = batch['input_ids'].to(DEVICE)
        attention_mask = batch['attention_mask'].to(DEVICE)
        pixel_values   = batch['pixel_values'].to(DEVICE)
        labels         = batch['labels'].to(DEVICE)
        
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            pixel_values=pixel_values,
            labels=labels
        )
        
        loss = outputs.loss / ACCUMULATION_STEPS
        loss.backward()
        
        if (i + 1) % ACCUMULATION_STEPS == 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            optimizer.zero_grad()  
            
        total_loss += loss.item() * ACCUMULATION_STEPS
        n_batches  += 1
        
        
        progress.set_postfix({'loss': f'{loss.item() * ACCUMULATION_STEPS:.4f}'})
        
    return total_loss / n_batches
    


def evaluate(model, dataloader, epoch_num):
    """
    Valuta il modello sul validation/test set.
    Ritorna la loss media.
    """
    model.eval()  # Disattiva dropout, modalità inferenza
    total_loss = 0.0
    n_batches  = 0
    
    progress = tqdm(dataloader, desc=f'Epoch {epoch_num} [Eval]')
    
    with torch.no_grad():  # Non calcolare gradienti — risparmia memoria
        for batch in progress:
            input_ids      = batch['input_ids'].to(DEVICE)
            attention_mask = batch['attention_mask'].to(DEVICE)
            pixel_values   = batch['pixel_values'].to(DEVICE)
            labels         = batch['labels'].to(DEVICE)
            
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                pixel_values=pixel_values,
                labels=labels
            )
            
            total_loss += outputs.loss.item()
            n_batches  += 1
            progress.set_postfix({'loss': f'{outputs.loss.item():.4f}'})
    
    return total_loss / n_batches


def save_checkpoint(model, epoch, train_loss, val_loss):
    """
    Salva i pesi LoRA alla fine di ogni epoca.
    Salva anche i metadati (loss, epoch) in un file JSON.
    """
    checkpoint_path = os.path.join(DIRS['checkpoints'], f'epoch_{epoch}')
    os.makedirs(checkpoint_path, exist_ok=True)
    
    # Salva solo i pesi LoRA (sono pochissimi MB, non tutto il modello)
    model.save_pretrained(checkpoint_path)
    
    # Salva i metadati
    metadata = {'epoch': epoch, 'train_loss': train_loss, 'val_loss': val_loss}
    with open(os.path.join(checkpoint_path, 'metadata.json'), 'w') as f:
        json.dump(metadata, f, indent=2)
    
    print(f'Checkpoint salvato: {checkpoint_path}')


def find_best_checkpoint():
    """
    Trova il checkpoint con la validation loss più bassa.
    Utile per riprendere il training o per l'inferenza.
    """
    checkpoints_dir = DIRS['checkpoints']
    best_loss = float('inf')
    best_path = None
    
    for epoch_dir in os.listdir(checkpoints_dir):
        metadata_path = os.path.join(checkpoints_dir, epoch_dir, 'metadata.json')
        if os.path.exists(metadata_path):
            with open(metadata_path) as f:
                meta = json.load(f)
            if meta['val_loss'] < best_loss:
                best_loss = meta['val_loss']
                best_path = os.path.join(checkpoints_dir, epoch_dir)
    
    return best_path, best_loss


def train(model, train_loader, test_loader, n_epochs=N_EPOCHS, lr=LEARNING_RATE):
    """
    Loop principale di training.
    Allena per N epoche e salva il checkpoint migliore.
    """
    # Solo i parametri LoRA vengono aggiornati
    trainable_params = [p for p in model.parameters() if p.requires_grad]
    optimizer = AdamW(trainable_params, lr=lr, weight_decay=0.01)
    
    history = {'train_loss': [], 'val_loss': []}
    best_val_loss = float('inf')
    
    print(f'Inizio training per {n_epochs} epoche...')
    print(f'Learning rate: {lr}')
    print('-' * 50)
    
    for epoch in range(1, n_epochs + 1):
        print(f'\n=== EPOCH {epoch}/{n_epochs} ===')
        
        # Training
        train_loss = train_one_epoch(model, train_loader, optimizer, epoch)
        
        # Valutazione
        val_loss = evaluate(model, test_loader, epoch)
        
        # Registra le loss
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        
        print(f'Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}')
        
        # Salva sempre il checkpoint
        save_checkpoint(model, epoch, train_loss, val_loss)
        
        # Salva il modello finale se è il migliore
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            model.save_pretrained(DIRS['final_model_v2'])
            print(f'Nuovo miglior modello salvato! Val Loss: {val_loss:.4f}')
    
    print('\nTraining completato!')
    print(f'Miglior Val Loss: {best_val_loss:.4f}')
    
    return history


print('Funzioni di training definite!')
print('\nPer avviare il training esegui la cella successiva.')

Funzioni di training definite!

Per avviare il training esegui la cella successiva.


In [11]:
# --- AVVIA IL TRAINING ---
# Questo richiede diverse ore. Lascialo girare di notte.

history = train(model, train_loader, test_loader, n_epochs=N_EPOCHS)

# Salva la history su disco
with open(os.path.join(BASE_PATH, 'training_history.json'), 'w') as f:
    json.dump(history, f, indent=2)

Inizio training per 1 epoche...
Learning rate: 2e-05
--------------------------------------------------

=== EPOCH 1/1 ===


Epoch 1 [Eval]: 100%|██████████| 800/800 [15:07<00:00,  1.13s/it, loss=3.8446]


Train Loss: 4.6570 | Val Loss: 3.8644
Checkpoint salvato: /kaggle/working/checkpoints/epoch_1
Nuovo miglior modello salvato! Val Loss: 3.8644

Training completato!
Miglior Val Loss: 3.8644


## STEP 9 — Valutazione qualitativa

In [12]:
from evaluate import load

def generate_answer(model, processor, image, question, max_new_tokens=15):
    """
    Genera una risposta usando la Greedy Search, 
    ottimizzata per risposte brevi (VQA).
    """
    model.eval()
    
    # Prompt rigoroso per LLaVA
    prompt = f"USER: <image>\n{question}\nASSISTANT:"
    
    inputs = processor(
        text=prompt,
        images=image,
        return_tensors='pt'
    ).to(DEVICE)
    
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,  
            do_sample=False,                
            pad_token_id=processor.tokenizer.pad_token_id
        )
    
    # Decodifica solo i nuovi token
    new_tokens = output_ids[0][inputs['input_ids'].shape[1]:]
    answer = processor.tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

    answer = answer.split("ASSISTANT:")[-1].strip()
    answer = answer.split("USER:")[0].strip()
    
    return answer


def evaluate_on_test_set(model, processor, flickr_test, test_vqa, n_samples=50):
    """
    Valuta il modello esclusivamente tramite BERTScore per una 
    valutazione puramente semantica e concettuale.
    """
    # Campiona casualmente N esempi
    samples = random.sample(test_vqa, min(n_samples, len(test_vqa)))
    
    generated_list = []
    expected_list = []
    results = []
    
    # 1. Generazione delle risposte da parte di LLaVA
    for sample in tqdm(samples, desc='Generazione risposte (Beam Search)'):
        image    = flickr_test[sample['image_idx']]['image']
        question = sample['question']
        expected = sample['answer']
        
        # Genera la risposta usando la funzione con Beam Search impostata prima
        generated = generate_answer(model, processor, image, question)
        
        # Accumuliamo i testi (convertiti in minuscolo per uniformità)
        generated_list.append(generated.lower())
        expected_list.append(expected.lower())
        
        results.append({
            'question':  question,
            'expected':  expected,
            'generated': generated
        })
    
    # 2. Calcolo parallelo del BERTScore su GPU
    print("\nCalcolo del BERTScore semantico...")
    bertscore_metric = load("bertscore")
    
    score_results = bertscore_metric.compute(
        predictions=generated_list, 
        references=expected_list, 
        lang="en"  # Specifica la lingua del dataset (Flickr30k è in inglese)
    )
    
    # Estraiamo l'F1-Score, l'indicatore più bilanciato tra precisione e richiamo
    f1_scores = score_results['f1']
    
    # Abbiniamo ogni punteggio al rispettivo esempio
    for idx, r in enumerate(results):
        r['bertscore'] = f1_scores[idx]
    
    # 3. Report Finale
    print(f'\n=== RISULTATI VALUTAZIONE SEMANTICA ({n_samples} esempi) ===')
    print(f'BERTScore F1 Medio: {np.mean(f1_scores):.4f}')
    print('Nota: Un punteggio > 0.85 indica un\'ottima corrispondenza concettuale.')
    
    # Mostra i primi 5 esempi reali per un controllo visivo
    print('\n--- Esempi di Verifica ---')
    for r in results[:5]:
        print(f'Q: {r["question"]}')
        print(f'Expected (Reale):  {r["expected"]}')
        print(f'Generated (IA):    {r["generated"]}')
        print(f'BERTScore:         {r["bertscore"]:.4f}')
        print()
    
    return results


# Esegui la valutazione
eval_results = evaluate_on_test_set(
    model, processor, dataset['test'], test_vqa, n_samples=150
)

Generazione risposte (Beam Search): 100%|██████████| 150/150 [04:58<00:00,  1.99s/it]



Calcolo del BERTScore semantico...


config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



=== RISULTATI VALUTAZIONE SEMANTICA (150 esempi) ===
BERTScore F1 Medio: 0.8901
Nota: Un punteggio > 0.85 indica un'ottima corrispondenza concettuale.

--- Esempi di Verifica ---
Q: How many people are mentioned?
Expected (Reale):  Two
Generated (IA):    Two men
ASSISTical: One man is looking at another man.
BERTScore:         0.8337

Q: What color is the man's shirt?
Expected (Reale):  White
Generated (IA):    White
ASIDE: The man is wearing a black shirt.
BERTScore:         0.8548

Q: Where is the scene set?
Expected (Reale):  In a nighttime environment
Generated (IA):    Outside a building with a group of children.
BERTScore:         0.9010

Q: Who is eating cake and strawberries?
Expected (Reale):  A little girl
Generated (IA):    A little girl with a blue dress on her shoulder.
BERTScore:         0.9248

Q: What is the woman doing?
Expected (Reale):  Dancing in a parade
Generated (IA):    Dancing in a costume with a green and yellow headpiece.
BERTScore:         0.9190



## **LIVE DEMO & COMPARISON**


## STEP 10 — Demo con Gradio

In [13]:
# # =====================================================================
# # SEZIONE DEMO: CONFRONTO TRA MODELLO BASE E MODELLO FINE-TUNED (LoRA)
# # =====================================================================

# import torch
# from transformers import LlavaProcessor, LlavaForConditionalGeneration, BitsAndBytesConfig
# import gradio as gr
# from PIL import Image

# # Configurazione Repository
# BASE_MODEL_ID = "llava-hf/llava-1.5-7b-hf"
# LORA_MODEL_ID = "Eikichi22/vqa-flickr30k-llava" # <--- METTI IL TUO USERNAME HF!

# DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
# print(f"Caricamento modello su {DEVICE}...")

# # 1. Caricamento del Processor e del Modello Base Quantizzato in 4-bit
# processor = LlavaProcessor.from_pretrained(BASE_MODEL_ID)
# bnb_config = BitsAndBytesConfig(
#     load_in_4bit=True,
#     bnb_4bit_quant_type='nf4',
#     bnb_4bit_compute_dtype=torch.float16
# )
# model = LlavaForConditionalGeneration.from_pretrained(
#     BASE_MODEL_ID,
#     quantization_config=bnb_config,
#     device_map='auto'
# )

# # 2. INIEZIONE NATIVA DELL'ADAPTER (La soluzione al bug!)
# print("Iniezione dei pesi LoRA...")
# model.load_adapter(LORA_MODEL_ID, adapter_name="flickr_lora")
# model.eval()
# print("[OK] Modello unificato pronto! (Adapter caricato con successo)")

# # 3. Funzione di Generazione Unificata (Corretta con la "s")
# def generate_vqa(model_type, image, question):
#     if not isinstance(image, Image.Image):
#         image = Image.fromarray(image)
#     image.thumbnail((1024, 1024))
#     image = image.convert("RGB")
    
#     prompt = f"USER: <image>\n{question}\nASSISTANT:"
#     inputs = processor(text=prompt, images=image, return_tensors='pt').to(DEVICE)
    
#     # 🔥 Lo switch corretto (con la "s" finale e senza il "with")
#     if model_type == "Base":
#         model.disable_adapters()  # Spegne l'adapter per la risposta base
#     else:
#         model.enable_adapters()   # Riaccende l'adapter per la risposta Fine-Tuned
        
#     # Ora che l'interruttore è nella posizione giusta, generiamo!
#     with torch.no_grad():
#         output_ids = model.generate(
#             **inputs,
#             max_new_tokens=15, 
#             do_sample=False,
#             pad_token_id=processor.tokenizer.pad_token_id
#         )
            
#     new_tokens = output_ids[0][inputs['input_ids'].shape[1]:]
#     answer = processor.tokenizer.decode(new_tokens, skip_special_tokens=True)
    
#     answer = answer.split("ASSISTANT:")[-1].strip()
#     answer = answer.split("USER:")[0].strip()
#     return answer

# # Funzioni target per i due bottoni in Gradio
# def predict_base(image, question):
#     if image is None or not question: return "Input mancanti."
#     return generate_vqa("Base", image, question)

# def predict_tuned(image, question):
#     if image is None or not question: return "Input mancanti."
#     return generate_vqa("Fine-Tuned", image, question)

# # 4. Interfaccia Gradio Parallela
# with gr.Blocks(title='VQA Evaluation: Base vs QLoRA') as demo:
#     gr.Markdown('# 📊 Analisi Comparativa VQA: LLaVA Base vs LLaVA Fine-Tuned (Flickr30k)')
#     gr.Markdown(
#         "Questa demo confronta dinamicamente l'attivazione e disattivazione dei pesi LoRA. "
#         "A sinistra il modello senza gli adapter (Originale), a destra con gli adapter attivi."
#     )
    
#     with gr.Row():
#         with gr.Column(scale=1):
#             image_input = gr.Image(label='📷 Carica Immagine', type='pil')
#             question_input = gr.Textbox(label='❓ Domanda (English)', placeholder='Es: What color is the sweater?')
#             submit_btn = gr.Button('Interroga i Modelli 🚀', variant='primary')
            
#         with gr.Column(scale=1):
#             gr.Markdown("### 🟢 LLaVA Modello Base (Senza LoRA)")
#             answer_base = gr.Textbox(label='Risposta Modello Base', lines=3)
            
#         with gr.Column(scale=1):
#             gr.Markdown("### 🔵 LLaVA Fine-Tuned (Con LoRA)")
#             answer_tuned = gr.Textbox(label='Risposta Modello QLoRA', lines=3)
            
#     submit_btn.click(fn=predict_base, inputs=[image_input, question_input], outputs=answer_base)
#     submit_btn.click(fn=predict_tuned, inputs=[image_input, question_input], outputs=answer_tuned)
    
#     question_input.submit(fn=predict_base, inputs=[image_input, question_input], outputs=answer_base)
#     question_input.submit(fn=predict_tuned, inputs=[image_input, question_input], outputs=answer_tuned)

# demo.launch(share=True)